# Similar Movies Debug Runner

Run the setup cell once, set `tmdb_ids`, then execute the result cell to inspect the standalone similar-movies flow with lane labels.

## Cell 1 - Setup

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from db.postgres import fetch_movie_cards, pool as postgres_pool
from search_v2.similar_movies import run_similar_movies_for_ids

if postgres_pool._closed:
    await postgres_pool.open()

print(f"Project root: {PROJECT_ROOT}")
print("Postgres pool open")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/michaelkeohane/Documents/movie-finder-rag
Postgres pool open


## Cell 2 - Anchors

## Cell 3 - Results

In [3]:
tmdb_ids = [6795]
limit = 25

def year_from_release_ts(release_ts: int | None) -> int | None:
    if release_ts is None:
        return None
    return datetime.fromtimestamp(release_ts, tz=timezone.utc).year


result = await run_similar_movies_for_ids(tmdb_ids, limit=limit)

cards = await fetch_movie_cards([item.movie_id for item in result.ranked])
cards_by_id = {card["movie_id"]: card for card in cards}

rows = []
for rank, item in enumerate(result.ranked, start=1):
    card = cards_by_id.get(item.movie_id, {})
    lane_scores = item.evidence.lane_scores
    rows.append(
        {
            "rank": rank,
            "title": card.get("title", "<missing card>"),
            "year": year_from_release_ts(card.get("release_ts")),
            "tmdb_id": item.movie_id,
            "score": round(item.score, 4),
            "dominant_lane": item.evidence.dominant_lane,
            "lanes": ", ".join(item.evidence.candidate_sources),
            "shape": round(lane_scores.get("shape", 0.0), 4),
            "director": round(lane_scores.get("director", 0.0), 4),
            "franchise": round(lane_scores.get("franchise", 0.0), 4),
            "studio": round(lane_scores.get("studio", 0.0), 4),
            "source": round(lane_scores.get("source", 0.0), 4),
            "quality": round(lane_scores.get("quality", 0.0), 4),
        }
    )

print("anchors:", result.anchor_movie_ids)
print("active_anchor_types:", result.active_anchor_types)
print("lane_weights:", result.debug.normalized_lane_weights)
print("vector_space_weights:", result.debug.vector_space_weights)
display(pd.DataFrame(rows))

anchors: [6795]
active_anchor_types: ['standard_shape', 'source_material']
lane_weights: {'shape': 0.5656565656565656, 'franchise': 0.12121212121212122, 'source': 0.12121212121212122, 'quality': 0.030303030303030304, 'format': 0.04040404040404041, 'themes': 0.12121212121212122, 'cast': 0.0, 'specific_award': 0.0, 'rare_keyword': 1.0, 'director': 1.0, 'studio': 0.06}
vector_space_weights: {'anchor': 0.09278350515463918, 'plot_events': 0.051546391752577324, 'plot_analysis': 0.2061855670103093, 'viewer_experience': 0.2061855670103093, 'watch_context': 0.15463917525773196, 'narrative_techniques': 0.11340206185567012, 'production': 0.061855670103092786, 'reception': 0.11340206185567012}


,rank,title,year,tmdb_id,score,dominant_lane,lanes,shape,director,franchise,studio,source,quality
0,1,Jumanji,1995,8844,0.9530,shape,"shape, source, quality, format, themes, rare_k...",1.0000,0.0,0.0,0.0,0.3439,0.9023
1,2,Legends of the Hidden Temple,2016,424661,0.7262,shape,"shape, quality, format, themes, rare_keyword",0.9698,0.0,0.0,0.0,0.0000,0.5893
2,3,"Honey, I Shrunk the Kids",1989,9354,0.6044,shape,"shape, quality, format, themes, rare_keyword",0.4860,0.0,0.0,0.0,0.0000,0.9155
3,4,The Borrowers,1997,9449,0.5771,shape,"shape, quality, format, themes, rare_keyword",0.4427,0.0,0.0,0.0,0.0000,0.8649
4,5,Jumanji: Welcome to the Jungle,2017,353486,0.5762,shape,"shape, source, quality, format, themes, rare_k...",0.3709,0.0,0.0,0.0,0.3439,0.9221
5,6,Spy Kids 3-D: Game Over,2003,12279,0.5662,shape,"shape, quality, format, themes, rare_keyword",0.6234,0.0,0.0,0.0,0.0000,0.8779
6,7,Journey 2: The Mysterious Island,2012,72545,0.5597,shape,"shape, quality, format, themes, rare_keyword",0.5857,0.0,0.0,0.0,0.0000,0.8783
7,8,"Honey, We Shrunk Ourselves",1997,11425,0.5443,shape,"shape, quality, format, themes, rare_keyword",0.3761,0.0,0.0,0.0,0.0000,0.8362
8,9,Goosebumps,2015,257445,0.5313,shape,"shape, quality, format, themes, rare_keyword",0.5767,0.0,0.0,0.0,0.0000,0.9031
9,10,Spy Kids,2001,10054,0.5152,shape,"shape, quality, format, themes, rare_keyword",0.5298,0.0,0.0,0.0,0.0000,0.9153
